<img src="http://dask.readthedocs.io/en/latest/_images/dask_horizontal.svg"
     align="right"
     width="30%"
     alt="Dask logo\">


# Dask DataFrame —— 并行化的 pandas

用起来像 pandas API，但面向并行与分布式工作流。

`dask.dataframe` 模块的核心是一个“分块并行”的 `DataFrame` 对象：接口像 pandas，却能用于并行和分布式场景。一个 Dask `DataFrame` 由许多沿索引切开、驻留内存的 pandas `DataFrame` 组成。对 Dask `DataFrame` 的一次操作，会在这些组成它的 pandas `DataFrame` 上触发许多 pandas 操作，并会顾及可能的并行度与内存限制。


<img src="https://docs.dask.org/en/stable/_images/dask-dataframe.svg"
     align="right"
     width="30%"
     alt="Dask DataFrame is composed of pandas DataFrames"/>

**相关文档**

* [DataFrame 文档](https://docs.dask.org/en/latest/dataframe.html)
* [DataFrame 视频](https://youtu.be/AT2XtFehFSQ)
* [DataFrame API](https://docs.dask.org/en/latest/dataframe-api.html)
* [DataFrame 示例](https://examples.dask.org/dataframe.html)
* [pandas 文档](https://pandas.pydata.org/pandas-docs/stable/)

## 什么时候用 `dask.dataframe`

pandas 很适合放得进内存的表格数据。关于 pandas，有一条经验法则：

> “内存大概要有数据集大小的 5 到 10 倍”
>
> ~ Wes McKinney（2017）在 [10 things I hate about pandas](https://wesmckinney.com/blog/apache-arrow-pandas-internals/)

这里的“数据集大小”指的是 **磁盘上** 的大小。

当数据超过这条经验法则时，Dask 就开始有用了。

本 notebook 会使用纽约市航班数据。这个数据集大约只有 200MB，方便在合理时间内下载，但 `dask.dataframe` 可以扩展到 **远大于内存** 的数据集。


## 创建数据集


创建本 notebook 会用到的数据集：


In [ ]:
%run prep.py -d flights

## 搭建本地集群


创建一个本地 Dask 集群，并把它连到 client。这段代码先不用深究，Distributed 那一节会详细讲。


In [ ]:
from dask.distributed import Client

client = Client(n_workers=4)
client

### Dask 诊断仪表盘

Dask Distributed 提供了很有用的 Dashboard，用来观察集群状态和计算过程。

如果你在 **JupyterLab 或 Binder** 上，可以用 [Dask JupyterLab 扩展](https://github.com/dask/dask-labextension)（环境里应该已经装好了）打开仪表盘图表：
* 点击左侧边栏的 Dask 图标
* 点击放大镜图标，它会自动连到当前活动的仪表盘（如果不行，可以把仪表盘链接 http://127.0.0.1:8787 填进输入框）
* 点击 **"Task Stream"**、**"Progress Bar"** 和 **"Worker Memory"**，这些图会在新标签页打开
* 按你的习惯重新排列标签页！

或者，点击上面 Client 详情里显示的仪表盘链接：http://127.0.0.1:8787/status。它会在新的浏览器标签页打开 Dashboard。


## 读取并使用数据集

我们来读取美国若干年份的航班摘录。这些数据特指从纽约市三个机场出发的航班。


In [ ]:
import os
import dask

按惯例，我们把模块 `dask.dataframe` 导入为 `dd`，并把对应的 `DataFrame` 对象叫做 `ddf`。

**说明**：“Dask DataFrame” 这个词稍微有点超载。视上下文，它可能指模块，也可能指 DataFrame 对象。为避免混淆，本 notebook 里约定：
- `dask.dataframe`（全小写）指 API，
- `DataFrame`（驼峰）指对象。

下面的文件名包含通配符 `*`，因此路径下所有匹配的文件都会读进同一个 `DataFrame`。


In [ ]:
import dask.dataframe as dd

ddf = dd.read_csv(
    os.path.join("data", "nycflights", "*.csv"), parse_dates={"Date": [0, 1, 2]}
)
ddf

Dask 此时还没有真正加载数据，它只是：
- 检查了输入路径，发现有十个匹配的文件
- 聪明地为每个分块创建了一组任务——在这个例子里，每个原始 CSV 文件对应一个任务


注意 `DataFrame` 对象的显示里还没有数据——Dask 只读了第一个文件的开头，用来推断列名和 dtype。


### 惰性求值（Lazy Evaluation）

包括 Dask `DataFrame` 在内，大多数 Dask 集合都是惰性求值的：Dask 会立刻构建计算的逻辑（也就是任务图），但只在必要时才真正“求值”。你可以用 `.visualize()` 查看这张任务图。

Delayed 那一节会对此讲得更多。现在只要记住：需要调用 `.compute()` 才会触发真正的计算。


In [ ]:
ddf.visualize()

有些函数（例如 `len` 和 `head`）也会触发计算。具体来说，调用 `len` 会：
- 加载实际数据（也就是把每个文件读进一个 pandas DataFrame）
- 再对每个 pandas DataFrame（也叫分区，partition）应用相应函数
- 把各部分的小计合并成最终总数


In [ ]:
# 加载数据并统计行数
len(ddf)


你可以像在 pandas 里那样查看数据的开头和结尾：


In [ ]:
ddf.head()

```python
ddf.tail()

# ValueError: Mismatched dtypes found in `pd.read_csv`/`pd.read_table`.

# +----------------+---------+----------+
# | Column         | Found   | Expected |
# +----------------+---------+----------+
# | CRSElapsedTime | float64 | int64    |
# | TailNum        | object  | float64  |
# +----------------+---------+----------+

# The following columns also raised exceptions on conversion:

# - TailNum
#   ValueError("could not convert string to float: 'N54711'")

# Usually this is due to dask's dtype inference failing, and
# *may* be fixed by specifying dtypes manually by adding:

# dtype={'CRSElapsedTime': 'float64',
#        'TailNum': 'object'}

# to the call to `read_csv`/`read_table`.

```


`pandas.read_csv` 会先读完整份文件再推断类型，而 `dask.dataframe.read_csv` 只从文件开头（若使用通配符，则是第一个文件）抽样。推断出的类型随后会强制应用到所有分区。

这个例子里，样本推断出的类型是错的。前 `n` 行的 `CRSElapsedTime` 没有值（pandas 会推断成 `float`），后面却变成了字符串（`object` dtype）。注意 Dask 给出了关于类型不匹配的明确报错。遇到这种情况，你有几种选择：

- 用 `dtype` 关键字直接指定类型。这是推荐做法，既最不容易出错（明确优于隐式），通常也最快。
- 增大 `sample` 关键字（按字节计）
- 使用 `assume_missing`，让 `dask` 假定被推断为 `int`（不允许缺失值）的列其实是 `float`（允许缺失值）。我们这个例子并不适用。

这里我们用第一种做法，直接指定出问题的列的 `dtype`。


In [ ]:
ddf = dd.read_csv(
    os.path.join("data", "nycflights", "*.csv"),
    parse_dates={"Date": [0, 1, 2]},
    dtype={"TailNum": str, "CRSElapsedTime": float, "Cancelled": bool},
)

In [ ]:
ddf.tail()  # 现在可以了


### 从远程存储读取

如果你在考虑分布式计算，数据多半存在远程服务上（例如 Amazon S3 或 Google 云存储），而且格式更友好（例如 Parquet）。Dask 可以从这些远程位置 **惰性且并行地** 直接读取多种格式。

下面是从 Amazon S3 读取纽约出租车数据的写法：

```python
ddf = dd.read_parquet(
    "s3://nyc-tlc/trip data/yellow_tripdata_2012-*.parquet",
)
```

你还可以利用 Parquet 特有的优化，例如列选择和元数据处理。更多内容见 [Dask 文档中关于 Parquet 的说明](https://docs.dask.org/en/stable/dataframe-parquet.html)。


## 用 `dask.dataframe` 做计算

我们来计算航班延误的最大值。

如果只用 pandas，我们会循环处理每个文件，找出各自的最大值，再在这些最大值里取总的最大。

```python
import pandas as pd

files = os.listdir(os.path.join('data', 'nycflights'))

maxes = []

for file in files:
    df = pd.read_csv(os.path.join('data', 'nycflights', file))
    maxes.append(df.DepDelay.max())
    
final_max = max(maxes)
```

`dask.dataframe` 让我们能写类似 pandas 的代码，在大于内存的数据集上并行运算。


In [ ]:
%%time
result = ddf.DepDelay.max()
result.compute()

这会为我们创建惰性计算，然后执行它。


**注意：** Dask 会尽快删除中间结果（例如每个文件对应的完整 pandas DataFrame）。因此你可以处理大于内存的数据集，但重复计算每次都得重新加载全部数据。（再跑一遍上面的代码，它比你预期的更快还是更慢？）


可以用 `.visualize()` 查看底层任务图：


In [ ]:
# 注意其中的并行
result.visualize()


## 练习

这一节你会做几个 `dask.dataframe` 计算。如果熟悉 pandas，这些应该不陌生。你需要想清楚何时调用 `.compute()`。


### 1. 数据集里有多少行？

_提示_：你会怎么检查一个列表里有多少元素？


In [ ]:
# 在此编写代码


In [ ]:
len(ddf)

### 2. 总共有多少次未取消的航班？

_提示_：使用 [布尔索引](https://pandas.pydata.org/pandas-docs/stable/indexing.html#boolean-indexing)。


In [ ]:
# 在此编写代码


In [ ]:
len(ddf[~ddf.Cancelled])

### 3. 每个机场总共有多少次未取消的航班？

*提示*：使用 [groupby](https://pandas.pydata.org/pandas-docs/stable/groupby.html)。


In [ ]:
# 在此编写代码


In [ ]:
ddf[~ddf.Cancelled].groupby("Origin").Origin.count().compute()

### 4. 每个机场的平均起飞延误是多少？


In [ ]:
# 在此编写代码


In [ ]:
ddf.groupby("Origin").DepDelay.mean().compute()

### 5. 一周中哪一天的平均起飞延误最严重？


In [ ]:
# 在此编写代码


In [ ]:
ddf.groupby("DayOfWeek").DepDelay.mean().idxmax().compute()

### 6. 假设 distance 列有误，你需要给所有值加 1，该怎么做？


In [ ]:
# 在此编写代码


In [ ]:
ddf["Distance"].apply(
    lambda x: x + 1
).compute()  # 先不用担心这个警告，下一节会讨论

# 或者

(ddf["Distance"] + 1).compute()


## 共享中间结果

做上面这些计算时，有时会对同一操作算了不止一次。对大多数操作，`dask.dataframe` 会保存参数，从而让重复计算被共享，只真正算一次。

例如，我们来计算所有未取消航班起飞延误的均值和标准差。因为 Dask 操作是惰性的，这些值还不是最终结果，只是得到结果所需的步骤。

如果用两次 `compute` 分别计算，中间计算不会被共享。


In [ ]:
non_canceled = ddf[~ddf.Cancelled]
mean_delay = non_canceled.DepDelay.mean()
std_delay = non_canceled.DepDelay.std()

In [ ]:
%%time

mean_delay_res = mean_delay.compute()
std_delay_res = std_delay.compute()

### `dask.compute`


这次试着把两者一起传给一次 `compute` 调用。


In [ ]:
%%time

mean_delay_res, std_delay_res = dask.compute(mean_delay, std_delay)

使用 `dask.compute` 大约只要一半时间。因为调用 `dask.compute` 时，两个结果的任务图会被合并，共享的操作只做一次而不是两次。具体来说，`dask.compute` 只会各做一次：

- 对 `read_csv` 的调用
- 过滤（`df[~df.Cancelled]`）
- 一部分必要的归约（`sum`、`count`）


要看多个结果合并后的任务图（以及哪些部分是共享的），可以使用 `dask.visualize` 函数（你可能想加上 `filename='graph.pdf'` 把图存到磁盘，方便放大查看）：


In [ ]:
dask.visualize(mean_delay, std_delay, engine="cytoscape")

### `.persist()`

使用分布式调度器时（后续 notebook 会更详细地讲调度器），你可以把一些 _经常要用的数据_ 留在 _分布式内存_ 里。

`persist` 会生成 “Futures”（后面也会讲），并把它们存放在与输出相同的结构里。任何能放进内存的数据或计算都可以使用 `persist`。


如果你只想分析从 JFK 机场出发、且未被取消的航班，可以像上一节那样调用两次 compute：


In [ ]:
non_cancelled = ddf[~ddf.Cancelled]
ddf_jfk = non_cancelled[non_cancelled.Origin == "JFK"]

In [ ]:
%%time
ddf_jfk.DepDelay.mean().compute()
ddf_jfk.DepDelay.sum().compute()

也可以考虑把那一部分数据 persist 到内存里。

看仪表盘上的 “Graph” 图，红色方块表示作为 Future 存在内存里的 persist 数据。你还会注意到 Worker Memory（另一张仪表盘图）占用上升。


In [ ]:
ddf_jfk = ddf_jfk.persist()  # 立刻把控制权交还给你


In [ ]:
%%time
ddf_jfk.DepDelay.mean().compute()
ddf_jfk.DepDelay.std().compute()

在这份 persist 过的数据上做分析会更快，因为我们不用重复加载和筛选（未取消、从 JFK 出发）。


## 在 Dask DataFrame 上使用自定义代码

`dask.dataframe` 只覆盖了 pandas API 中较小但常用的一部分。

之所以有这个限制，有两个原因：

1.  Pandas 的 API *非常大*
2.  有些操作本身就很难并行，例如排序。

另外，像 `set_index` 这类重要操作虽然能用，但比 pandas 慢，因为它们包含大量数据洗牌（shuffle），还可能写磁盘。

**如果你想用一些 Dask DataFrame 还没实现（或没法实现）的自定义函数，该怎么办？**

可以在 [Dask issue tracker](https://github.com/dask/dask/issues) 开一个 issue，问问实现这个函数有多可行，也可以考虑把函数贡献给 Dask。

如果它是自定义函数，或实现起来很棘手，`dask.dataframe` 提供了几种方法，让自定义函数更容易作用在 Dask DataFrame 上：

- [`map_partitions`](https://docs.dask.org/en/latest/generated/dask.dataframe.DataFrame.map_partitions.html)：在 Dask DataFrame 的每个分区（每个 pandas DataFrame）上运行函数
- [`map_overlap`](https://docs.dask.org/en/latest/generated/dask.dataframe.rolling.map_overlap.html)：在每个分区上运行函数，并在相邻分区之间共享若干行
- [`reduction`](https://docs.dask.org/en/latest/generated/dask.dataframe.Series.reduction.html)：用于自定义的按行归约操作


我们快速看一下 `map_partitions()` 函数：


In [ ]:
help(ddf.map_partitions)

`ddf` 里的 "Distance" 列目前以英里为单位。假设我们想把它转成公里，并且已经有下面这个通用辅助函数。这时可以用 `map_partitions`，把函数并行应用到内部的每个 pandas `DataFrame` 上。


In [ ]:
import pandas as pd


def my_custom_converter(df, multiplier=1):
    return df * multiplier


meta = pd.Series(name="Distance", dtype="float64")

distance_km = ddf.Distance.map_partitions(
    my_custom_converter, multiplier=0.6, meta=meta
)


In [ ]:
distance_km.visualize()

In [ ]:
distance_km.head()

### 什么是 `meta`？

因为 Dask 是惰性执行的，它并不总有足够信息来推断某些操作的输出结构（包括数据类型）。

`meta` 是给 Dask 的一个 _提示_，说明你的计算输出长什么样。重要的是，`meta` _不会去干涉_ 最终的输出结构。在 Dask 能确定真实输出结构之前，它会先使用这个 `meta`。

定义 `meta` 的方法有很多，我们建议用一个很小的 pandas Series 或 DataFrame，使其结构与最终输出一致。


## 关闭本地 Dask 集群


养成习惯：自己创建的 Dask 集群用完后都关掉。


In [ ]:
client.shutdown()